In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
bn_df = pd.read_csv('bn.csv')
en_df = pd.read_csv('en.csv')

In [3]:
bn_df.head()

,category,headline,content,label
0,National,৮ দিনে ১৮ বিল পাস!,দশম জাতীয় সংসদের মেয়াদ শেষ হয়ে যাচ্ছে। কার্যত ...,0.0
1,Sports,আ’লীগের জনসভায় লোকে লোকারণ্য ফেনী ট্রাংক রোড,একাদশ জাতীয় সংসদ নির্বাচনকে সামনে রেখে সাংগঠনি...,0.0
2,National,মাদ্রাসায় জোড়া খুন: পরিচালক তিন দিনের রিমান্ডে,গাজীপুরে জোড়া খুন মামলার প্রধান আসামি মাদ্রাসা...,0.0
3,Sports,নেপালকে হারিয়ে গ্রুপ চ্যাম্পিয়ন বাংলাদেশ,সাফ অনূর্ধ্ব-১৮ নারী ফুটবল চ্যাম্পিয়নশিপে নেপা...,1.0
4,National,কুড়িগ্রামে ২ শিক্ষার্থীর লাশ উদ্ধার,কুড়িগ্রাম প্রতিনিধি : কুড়িগ্রাম সদর উপজেলার বে...,1.0


In [4]:
en_df.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [5]:
bn_df = bn_df.drop(columns=['category'])
bn_df['label'] = bn_df['label'].astype(int)

bn_df.head()

,headline,content,label
0,৮ দিনে ১৮ বিল পাস!,দশম জাতীয় সংসদের মেয়াদ শেষ হয়ে যাচ্ছে। কার্যত ...,0
1,আ’লীগের জনসভায় লোকে লোকারণ্য ফেনী ট্রাংক রোড,একাদশ জাতীয় সংসদ নির্বাচনকে সামনে রেখে সাংগঠনি...,0
2,মাদ্রাসায় জোড়া খুন: পরিচালক তিন দিনের রিমান্ডে,গাজীপুরে জোড়া খুন মামলার প্রধান আসামি মাদ্রাসা...,0
3,নেপালকে হারিয়ে গ্রুপ চ্যাম্পিয়ন বাংলাদেশ,সাফ অনূর্ধ্ব-১৮ নারী ফুটবল চ্যাম্পিয়নশিপে নেপা...,1
4,কুড়িগ্রামে ২ শিক্ষার্থীর লাশ উদ্ধার,কুড়িগ্রাম প্রতিনিধি : কুড়িগ্রাম সদর উপজেলার বে...,1


In [6]:
en_df = en_df.drop(en_df.columns[0], axis=1)
en_df = en_df.rename(columns={'title': 'headline', 'text': 'content'})
en_df['label'] = en_df['label'].astype(int)

en_df.head()

,headline,content,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [7]:
bn_df = bn_df.dropna()
bn_df = bn_df.drop_duplicates()
bn_df.shape

(14534, 3)

In [8]:

en_df = en_df.dropna()
en_df = en_df.drop_duplicates()
en_df.shape

(63121, 3)

In [9]:
en_df['headline'] = en_df['headline'].str.lower()
en_df['content'] = en_df['content'].str.lower()
en_df.head()

,headline,content,label
0,law enforcement on high alert following threat...,no comment is expected from barack obama membe...,1
2,unbelievable! obama’s attorney general says mo...,"now, most of the demonstrators gathered last ...",1
3,"bobby jindal, raised hindu, uses story of chri...",a dozen politically active pastors came here f...,0
4,satan 2: russia unvelis an image of its terrif...,"the rs-28 sarmat missile, dubbed satan 2, will...",1
5,about time! christian group sues amazon and sp...,all we can say on this one is it s about time ...,1


In [10]:
def clean_text(text):
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [11]:
bn_df['headline'] = bn_df['headline'].apply(clean_text)
bn_df['content'] = bn_df['content'].apply(clean_text)

bn_df.head()

,headline,content,label
0,৮ দিনে ১৮ বিল পাস!,দশম জাতীয় সংসদের মেয়াদ শেষ হয়ে যাচ্ছে। কার্যত ...,0
1,আ’লীগের জনসভায় লোকে লোকারণ্য ফেনী ট্রাংক রোড,একাদশ জাতীয় সংসদ নির্বাচনকে সামনে রেখে সাংগঠনি...,0
2,মাদ্রাসায় জোড়া খুন: পরিচালক তিন দিনের রিমান্ডে,গাজীপুরে জোড়া খুন মামলার প্রধান আসামি মাদ্রাসা...,0
3,নেপালকে হারিয়ে গ্রুপ চ্যাম্পিয়ন বাংলাদেশ,সাফ অনূর্ধ্ব-১৮ নারী ফুটবল চ্যাম্পিয়নশিপে নেপা...,1
4,কুড়িগ্রামে ২ শিক্ষার্থীর লাশ উদ্ধার,কুড়িগ্রাম প্রতিনিধি : কুড়িগ্রাম সদর উপজেলার বে...,1


In [12]:
en_df['headline'] = en_df['headline'].apply(clean_text)
en_df['content'] = en_df['content'].apply(clean_text)

en_df.head()

,headline,content,label
0,law enforcement on high alert following threat...,no comment is expected from barack obama membe...,1
2,unbelievable! obama’s attorney general says mo...,"now, most of the demonstrators gathered last n...",1
3,"bobby jindal, raised hindu, uses story of chri...",a dozen politically active pastors came here f...,0
4,satan 2: russia unvelis an image of its terrif...,"the rs-28 sarmat missile, dubbed satan 2, will...",1
5,about time! christian group sues amazon and sp...,all we can say on this one is it s about time ...,1


In [13]:
BN_X = bn_df[['headline', 'content']]
BN_Y = bn_df['label']

EN_X = en_df[['headline', 'content']]
EN_Y = en_df['label']

In [14]:
BN_X_TRAIN, BN_X_TEST, BN_Y_TRAIN, BN_Y_TEST = train_test_split(BN_X, BN_Y, test_size=0.2, random_state=42, stratify=BN_Y)
EN_X_TRAIN, EN_X_TEST, EN_Y_TRAIN, EN_Y_TEST = train_test_split(EN_X, EN_Y, test_size=0.2, random_state=42, stratify=EN_Y)

In [15]:
BN_X_TRAIN['text'] = BN_X_TRAIN['headline'] + ' ' + BN_X_TRAIN['content']
BN_X_TEST['text'] = BN_X_TEST['headline'] + ' ' + BN_X_TEST['content']

EN_X_TRAIN['text'] = EN_X_TRAIN['headline'] + ' ' + EN_X_TRAIN['content']
EN_X_TEST['text'] = EN_X_TEST['headline'] + ' ' + EN_X_TEST['content']

In [16]:
en_vectorizer = TfidfVectorizer(max_features=10000)
bn_vectorizer = TfidfVectorizer(max_features=10000)

bn_vectorizer.fit(BN_X_TRAIN['text'])
en_vectorizer.fit(EN_X_TRAIN['text'])

BN_X_TRAIN_TFIDF = bn_vectorizer.transform(BN_X_TRAIN['text'])
BN_X_TEST_TFIDF = bn_vectorizer.transform(BN_X_TEST['text'])

EN_X_TRAIN_TFIDF = en_vectorizer.transform(EN_X_TRAIN['text'])
EN_X_TEST_TFIDF = en_vectorizer.transform(EN_X_TEST['text'])

In [17]:
bn_model = LogisticRegression(max_iter=1000)
bn_model.fit(BN_X_TRAIN_TFIDF, BN_Y_TRAIN)

en_model = LogisticRegression(max_iter=1000)
en_model.fit(EN_X_TRAIN_TFIDF, EN_Y_TRAIN)

LogisticRegression(max_iter=1000)

In [18]:
import pickle

with open('bn_model.pkl', 'wb') as f:
    pickle.dump(bn_model, f)

with open('en_model.pkl', 'wb') as f:
    pickle.dump(en_model, f)

with open('bn_vectorizer.pkl', 'wb') as f:
    pickle.dump(bn_vectorizer, f)

with open('en_vectorizer.pkl', 'wb') as f:
    pickle.dump(en_vectorizer, f)

In [19]:
BN_Y_PRED = bn_model.predict(BN_X_TEST_TFIDF)
accuracy_score(BN_Y_TEST, BN_Y_PRED)

0.7506019951840385

In [20]:
EN_Y_PRED = en_model.predict(EN_X_TEST_TFIDF)
accuracy_score(EN_Y_TEST, EN_Y_PRED)

0.9502574257425742

In [21]:
import time
import os
import joblib
from datetime import datetime
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

X_sample = BN_X_TEST_TFIDF[:100]

results = []

def log_result(model_name, lang, y_true, y_pred, y_proba, model_path, train_time=None, model=None, scaler=None):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')

    if y_proba.shape[1] > 1:
        roc = roc_auc_score(y_true, y_proba[:, 1])
    else:
        roc = roc_auc_score(y_true, y_pred)

    if model is not None and scaler is not None:
        X_test_sample = scaler.transform(X_sample)
        start = time.time()
        for _ in range(100):
            model.predict(X_test_sample)
        inf_time_ms = (time.time() - start) / 100 * 1000
    else:
        start = time.time()
        for _ in range(100):
            model.predict(X_sample)
        inf_time_ms = (time.time() - start) / 100 * 1000

    size_mb = os.path.getsize(model_path) / (1024 * 1024)

    results.append({
        "model": model_name,
        "language": lang,
        "accuracy": round(acc, 4),
        "precision": round(prec, 4),
        "recall": round(rec, 4),
        "f1_macro": round(f1, 4),
        "roc_auc": round(roc, 4),
        "inference_ms_per_sample": round(inf_time_ms, 2),
        "model_size_mb": round(size_mb, 2),
        "train_time_s": round(train_time, 1) if train_time else None,
        "artifact": os.path.basename(model_path),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M")
    })

In [22]:
import os

os.makedirs('experiments', exist_ok=True)

for lang, prefix in [("Bengali", "bn"), ("English", "en")]:
    X_train = globals()[f"{prefix.upper()}_X_TRAIN_TFIDF"]
    y_train = globals()[f"{prefix.upper()}_Y_TRAIN"]
    X_test  = globals()[f"{prefix.upper()}_X_TEST_TFIDF"]
    y_test  = globals()[f"{prefix.upper()}_Y_TEST"]

    start = time.time()
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    path = f"experiments/{prefix}_logreg.pkl"
    joblib.dump(model, path)

    log_result("LogisticRegression", lang, y_test, y_pred, y_proba, path,
                train_time, model=model)

In [23]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
import joblib
import time

for lang, prefix in [("Bengali", "bn"), ("English", "en")]:
    print(f"Training LinearSVM for {lang}...")

    X_train = globals()[f"{prefix.upper()}_X_TRAIN_TFIDF"]
    X_test  = globals()[f"{prefix.upper()}_X_TEST_TFIDF"]
    y_train = globals()[f"{prefix.upper()}_Y_TRAIN"]
    y_test  = globals()[f"{prefix.upper()}_Y_TEST"]

    scaler = StandardScaler(with_mean=False)
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    start = time.time()

    svm_base = LinearSVC(
        C=0.5,
        penalty='l2',
        loss='squared_hinge',
        dual=True,
        tol=1e-4,
        max_iter=50000,
        random_state=42,
        class_weight='balanced'
    )

    svm_base.fit(X_train_s, y_train)

    model = CalibratedClassifierCV(
        svm_base,
        method='sigmoid',
        cv='prefit'
    )
    model.fit(X_train_s, y_train)

    train_time = time.time() - start

    y_pred  = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)

    path = f"experiments/{prefix}_svm.pkl"
    joblib.dump((model, scaler), path)

    log_result(
        model_name="LinearSVM",
        lang=lang,
        y_true=y_test,
        y_pred=y_pred,
        y_proba=y_proba,
        model_path=path,
        train_time=train_time,
        model=model,
        scaler=scaler
    )

    print(f"LinearSVM {lang} done in {train_time:.1f}s — NO convergence warnings!")

Training LinearSVM for Bengali...


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


LinearSVM Bengali done in 47.9s — NO convergence warnings!
Training LinearSVM for English...


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


LinearSVM English done in 235.3s — NO convergence warnings!


In [25]:
from sklearn.ensemble import RandomForestClassifier

for lang, prefix in [("Bengali", "bn"), ("English", "en")]:
    print(f"Training RandomForest for {lang}...")

    X_train = globals()[f"{prefix.upper()}_X_TRAIN_TFIDF"]
    y_train = globals()[f"{prefix.upper()}_Y_TRAIN"]
    X_test  = globals()[f"{prefix.upper()}_X_TEST_TFIDF"]
    y_test  = globals()[f"{prefix.upper()}_Y_TEST"]

    start = time.time()
    rf = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=42
    )
    rf.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)

    path = f"experiments/{prefix}_rf.pkl"
    joblib.dump(rf, path)

    log_result(
        model_name="RandomForest",
        lang=lang,
        y_true=y_test,
        y_pred=y_pred,
        y_proba=y_proba,
        model_path=path,
        train_time=train_time,
        model=rf
    )

    print(f"RandomForest {lang} done in {train_time:.1f}s")

Training RandomForest for Bengali...
RandomForest Bengali done in 45.4s
Training RandomForest for English...
RandomForest English done in 225.2s


In [26]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["language", "f1_macro"], ascending=False)

results_df.to_csv("experiments/comparison_table.csv", index=False)
display(results_df)

print("\nBengali best model →", results_df[results_df.language=="Bengali"].iloc[0]["model"],
      "with F1 =", results_df[results_df.language=="Bengali"].iloc[0]["f1_macro"])
print("English best model →", results_df[results_df.language=="English"].iloc[0]["model"],
      "with F1 =", results_df[results_df.language=="English"].iloc[0]["f1_macro"])

,model,language,accuracy,precision,recall,f1_macro,roc_auc,inference_ms_per_sample,model_size_mb,train_time_s,artifact,timestamp
1,LogisticRegression,English,0.9503,0.9499,0.9496,0.9497,0.9888,0.94,0.08,6.0,en_logreg.pkl,2025-11-30 03:52
5,RandomForest,English,0.9381,0.9375,0.9373,0.9374,0.9873,75.09,348.19,225.2,en_rf.pkl,2025-11-30 04:03
3,LinearSVM,English,0.9285,0.9278,0.9276,0.9277,0.9780,0.44,0.31,235.3,en_svm.pkl,2025-11-30 03:56
0,LogisticRegression,Bengali,0.7506,0.7326,0.6392,0.6497,0.7775,0.96,0.08,14.3,bn_logreg.pkl,2025-11-30 03:52
2,LinearSVM,Bengali,0.6522,0.6020,0.6063,0.6036,0.6634,0.45,0.31,47.9,bn_svm.pkl,2025-11-30 03:52
4,RandomForest,Bengali,0.7262,0.7020,0.5934,0.5901,0.7501,62.69,172.44,45.4,bn_rf.pkl,2025-11-30 03:59



Bengali best model → LogisticRegression with F1 = 0.6497
English best model → LogisticRegression with F1 = 0.9497
